# RuinedFooocus + Face Swap (Colab, inline)

A fork of [runew0lf/RuinedFooocus](https://github.com/runew0lf/RuinedFooocus) with a **real InsightFace face swap** added, running **inside this notebook** — no `gradio.live` link, no public URL.

**Run the cells in order.** Each one tells you what it is doing.

| | |
|---|---|
| GPU | Runtime → Change runtime type → **T4** (or better) |
| Python | Colab's current runtime — **do not** change the runtime version |
| Disk | ~13 GB (models) |
| First run | ~10–15 min (installs + downloads) |
| Later runs | ~2 min |

> **Face swap note:** `inswapper_128.onnx` is InsightFace's *non-commercial research* model. Personal use is fine; commercial use needs a licence from InsightFace.


## 1. Check the machine

Confirms you have a GPU and shows the Python version. The dependency pins in this fork are built for **Python 3.12 and 3.13**, so no runtime downgrade is needed.


In [ ]:
import sys, platform, subprocess, os

print("Python :", sys.version.split()[0])
print("Machine:", platform.platform())

try:
    import torch
    print("torch  :", torch.__version__)
except ImportError:
    print("torch  : (will be installed by the app)")

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode == 0:
    print("GPU    :", smi.stdout.strip())
else:
    print("GPU    : NOT FOUND  <-- set Runtime > Change runtime type > T4 GPU")

free = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print()
print(free.stdout.strip())

## 2. Get the code

Clones **your fork**, so anything pushed later is picked up by re-running this cell.


In [ ]:
REPO = "https://github.com/ahmrafi22/RuinedFooocus.git"
APP_DIR = "/content/RuinedFooocus"

import os, subprocess, sys

if os.path.isdir(APP_DIR):
    print("Already cloned, updating...")
    subprocess.run(["git", "-C", APP_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, APP_DIR], check=True)

os.chdir(APP_DIR)
print("Working dir:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

## 3. Start the app (background)

`launch.py` installs torch, the pinned dependencies and ComfyUI, then **never returns** (`webui.py` ends in a `while True` loop). So it runs as a background process and the next cells just watch its log.

This cell returns as soon as the process is started — the install happens while you read on.


In [ ]:
import subprocess, os, sys

os.chdir("/content/RuinedFooocus")

# Colab already ships a CUDA torch; freezing stops the app from reinstalling it.
open("freezetorch", "w").close()

# --offline: skip the update check and the model auto-download at startup
# --nobrowser: there is no browser to open
# --listen 127.0.0.1: only reachable through the notebook's own port proxy
LOG = "/content/fooocus.log"
cmd = [
    sys.executable, "launch.py",
    "--offline", "--nobrowser",
    "--listen", "127.0.0.1",
    "--port", "7865",
]

env = dict(os.environ)
env["TORCH_PLATFORM"] = "cu124"        # matches Colab's preinstalled torch
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

log = open(LOG, "w")
proc = subprocess.Popen(
    cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd="/content/RuinedFooocus",
)

print("Started PID", proc.pid)
print("Log:", LOG)
print()
print("Wait for the next cell to report READY. First run installs torch deps and")
print("takes ~10-15 min; later runs are much faster.")

## 4. Wait for it to come up

Polls the log and the port. When it prints **READY**, run the next cell.

If it prints **FAILED**, the last 40 lines of the log are shown — that is the actual error, not a guess.


In [ ]:
import time, os, urllib.request

LOG = "/content/fooocus.log"
URL = "http://127.0.0.1:7865/"

deadline = time.time() + 60 * 25          # 25 min ceiling
last_len = 0
ready = False

while time.time() < deadline:
    if os.path.exists(LOG):
        text = open(LOG, encoding="utf-8", errors="ignore").read()
        if len(text) > last_len:
            for line in text[last_len:].splitlines():
                line = line.strip()
                if line and not line.startswith(("0%|", "1%|", "2%|", "3%|", "4%|", "5%|")):
                    print(line[:150])
            last_len = len(text)

        if "Starting webui" in text:
            pass

    if proc.poll() is not None:
        print("\nFAILED - the process exited with code", proc.returncode)
        print("-" * 60)
        print("\n".join(open(LOG, encoding="utf-8", errors="ignore").read().splitlines()[-40:]))
        break

    try:
        with urllib.request.urlopen(URL, timeout=3) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass

    time.sleep(5)

print()
if ready:
    print("READY -> run the next cell")
else:
    print("Not ready yet. If the log above shows no progress, re-run this cell.")

## 5. Show the UI here

Embeds the running app in an iframe **inside this notebook**, using Colab's own kernel port proxy. Nothing is exposed publicly.

If the frame stays blank, use the link it prints underneath — that is the same app, opened directly.


In [ ]:
from google.colab import output as colab_output

PORT = 7865

# Re-assert the proxy binding on every run, so re-running this cell recovers
# a frame that was closed.
colab_output.serve_kernel_port_as_iframe(PORT, path="/", height=900, width="100%")

print()
print("If the frame above is blank, open this in a new tab:")
print(f"    https://localhost:{PORT}/  (via Colab's port viewer)")
print()
print("Inside the app, use the 'Face Swap' tab:")
print("  1. upload the face you want in 'Source face image'")
print("  2. tick 'Enable face swap'")
print("  3. generate as usual")

## 6. Models

Nothing is downloaded until it is needed, and everything lives on the Colab VM's disk (lost when the runtime ends).

| Model | Where | Size |
|---|---|---|
| SDXL base 1.0 | `models/checkpoints/` | 6.9 GB |
| inswapper_128 (the swap) | `models/faceswap/` | 554 MB |
| buffalo_l (face detector) | `models/faceswap/insightface/` | 289 MB |

The swap models download themselves the first time you use the Face Swap tab. The checkpoint does not, so this cell fetches it.

> Prefer a different checkpoint? Drop any SDXL `.safetensors` into `models/checkpoints/` and pick it in the UI instead.


In [ ]:
import os, urllib.request

DEST = "/content/RuinedFooocus/models/checkpoints"
os.makedirs(DEST, exist_ok=True)

TARGET = os.path.join(DEST, "sd_xl_base_1.0_0.9vae.safetensors")
URL = ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"
       "resolve/main/sd_xl_base_1.0_0.9vae.safetensors")

if os.path.exists(TARGET) and os.path.getsize(TARGET) > 6e9:
    print("Already present:", TARGET)
else:
    print("Downloading SDXL base (~6.9 GB)...")

    def progress(blocks, block_size, total):
        done = blocks * block_size
        if total > 0 and blocks % 200 == 0:
            print(f"  {done/1e9:.1f} / {total/1e9:.1f} GB", end="\r")

    urllib.request.urlretrieve(URL, TARGET, progress)
    print()
    print("Saved:", TARGET, f"({os.path.getsize(TARGET)/1e9:.1f} GB)")

## 7. Pre-download the face swap models (optional)

Saves you waiting mid-generation. Uses the fork's own `modules/faceswap.py`, so it tests the exact code path the app uses.


In [ ]:
import os, sys
os.chdir("/content/RuinedFooocus")
sys.path.insert(0, "/content/RuinedFooocus")

import modules.faceswap as faceswap

faceswap.get_swapper()          # inswapper_128.onnx (554 MB, from the app's own registry)
faceswap.get_analyzer(640)      # buffalo_l (289 MB)

print()
print(faceswap.status())

## 8. Troubleshooting

**The UI frame is blank.** Run cell 5 again. Colab's proxy can drop the frame on a reconnect; the app itself keeps running.

**"No face found in the generated image."** The swap ran but the detector found nothing. Raise *Detector size* in the Face Swap tab, or use a generation where the face is larger in frame.

**The face looks pasted-on.** That is what a 128px inswapper paste looks like on a large head. Keep *Blend into the image* on, generate at a normal portrait framing, and raise *Detector size*.

**Everything is slow.** Check you are on a GPU (cell 1). On CPU it will not finish.

**The process died.** Re-run cell 3. The log at `/content/fooocus.log` has the real error.

**You changed the code on GitHub.** Re-run cell 2 (`git pull`), then re-run cell 3.

To stop the app: `proc.terminate()`
